In [ ]:
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install PyPDF2



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, Dataset


In [ ]:
data = {
    "Resume": [
        "Developed machine learning models using Python and scikit-learn.",
        "Built websites using HTML, CSS, JavaScript, and React.",
        "Managed project timelines and coordinated with teams.",
        "Implemented deep learning models and NLP pipelines."
    ],
    "Category": [
        "Data Scientist",
        "Web Developer",
        "Project Manager",
        "Data Scientist"
    ]
}
df = pd.DataFrame(data)


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])
num_labels = len(le.classes_)


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


In [ ]:
class ResumeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(text, return_tensors="pt", padding='max_length',
                                truncation=True, max_length=self.max_len)
        return {
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx])
        }


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['Resume'], df['label'], test_size=0.2, random_state=42)

train_dataset = ResumeDataset(X_train.tolist(), y_train.tolist(), tokenizer)
test_dataset = ResumeDataset(X_test.tolist(), y_test.tolist(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4)
test_loader = DataLoader(test_dataset, batch_size=4)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model.train()
for epoch in range(3):
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch+1} Loss: {loss.item()}")


Epoch 1 Loss: 1.434749960899353
Epoch 2 Loss: 1.0376142263412476
Epoch 3 Loss: 0.8974557518959045


In [ ]:
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, axis=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

print(classification_report(true_labels, predictions, labels=[0, 1, 2], target_names=le.classes_))



                 precision    recall  f1-score   support

 Data Scientist       0.00      0.00      0.00       0.0
Project Manager       0.00      0.00      0.00       0.0
  Web Developer       0.00      0.00      0.00       1.0

       accuracy                           0.00       1.0
      macro avg       0.00      0.00      0.00       1.0
   weighted avg       0.00      0.00      0.00       1.0



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/me

In [ ]:
def predict_resume(text):
    inputs = tokenizer(text, return_tensors="pt", padding='max_length',
                       truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    pred = torch.argmax(logits, axis=1)
    return le.inverse_transform(pred.cpu().numpy())[0]

# Test
print(predict_resume("Experience with Flask, MongoDB, and building REST APIs."))


Data Scientist


In [ ]:
from google.colab import files
uploaded = files.upload()
import PyPDF2

# Load and read the uploaded PDF
file_name = list(uploaded.keys())[0]  # Gets the actual filename of uploaded PDF
pdf_reader = PyPDF2.PdfReader(file_name)
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

print(text)
  # Print first 1000 characters of the resume text



Saving Updated resume.pdf to Updated resume (2).pdf
Rishu Agarwa l                                                                                                                             
Email - rishu.2426mca215@kiet.edu                Leetcode  - https://leetcode.com/u/Rishu_Agarwal /                                                          
Linkedin Id - rishu -agarwal -3a62992b4                   GITHUB - https://github.com/Rishu200h t                                                                                                   
 
EDUCATIO N   
Master of Computer Applications – MCA                                                                                          Sep/2024 - Present    
KIET Group of Institutions   
                                                    
Bachelor's of Science  (PCM)                                                                                                                 Aug/2020 - Sep/2023   
CCSU (Chaudhary Charan Singh University)